# Data Download

Downloads monthly stock prices and benchmark indices from Yahoo Finance for the 100-stock universe (50 US, 50 UK). Saves raw data to CSV and cleaned monthly data to parquet.


## Universe Definition

Defines the 100-stock universe split evenly between US and UK large-cap stocks.


In [19]:
from pathlib import Path
import pandas as pd
import yfinance as yf

# Benchmarks
SNP  = "^GSPC"
FTSE = "^FTSE"

# Date config
START = "2005-01-01"
END   = "2025-09-30"
FREQ  = "1mo"
AUTO_ADJUST = True

# Paths
SAVE_PATH      = Path("../data/raw")
INTERIM_PATH   = Path("../data/interim")
PROCESSED_PATH = Path("../data/processed")
for p in [SAVE_PATH, INTERIM_PATH, PROCESSED_PATH]:
    p.mkdir(parents=True, exist_ok=True)

# Universe
US_TICKERS = [
    # Originals (US subset)
    "AAPL","JPM","XOM","PFE","WMT",
    # Large US names (Yahoo symbols)
    "MSFT","AMZN","GOOGL","META","NVDA","V","MA","JNJ","PG","CVX","HD","BAC","PEP",
    "KO","ABBV","AVGO","COST","DIS","CSCO","TMO","MRK","ABT","NFLX","CRM","ACN",
    "AMD","INTC","QCOM","TXN","LIN","NKE","MCD","HON","CAT","UPS","PM","IBM",
    "ORCL","AMAT","BKNG","LLY","UNH","ADP","BA","GS"
]

UK_TICKERS = [
    # Originals (UK subset)
    "HSBA.L","BP.L","TSCO.L","AZN.L","RR.L",
    # FTSE-100 names
    "SHEL.L","GSK.L","BATS.L","ULVR.L","RIO.L","GLEN.L","RKT.L","LLOY.L","BARC.L","VOD.L",
    "BT-A.L","DGE.L","RTO.L","NG.L","SSE.L","PRU.L","SDR.L","SBRY.L","AV.L","AAL.L","STAN.L",
    "CPG.L","REL.L","AHT.L","IAG.L","PSN.L","TW.L","SGRO.L","LAND.L","FERG.L",
    "LGEN.L","ADM.L","AUTO.L","BNZL.L","EXPN.L","III.L","MNDI.L","OCDO.L","PSON.L",
    "SPX.L","WTB.L","JD.L","SGE.L", "SN.L", "SKG.L"
]

# Make a dataframe (add sector later)
uni = (pd.DataFrame({
    "ticker": list(dict.fromkeys(US_TICKERS + UK_TICKERS))  # de-dupe, keep order
})
.assign(country=lambda d: d["ticker"].str.endswith(".L").map({True:"UK", False:"US"}))
.assign(sector="Unknown"))

print(f"Universe size: {len(uni)} (US={sum(uni.country=='US')}, UK={sum(uni.country=='UK')})")

# Save the universe so later notebooks can use metadata if needed
UNI_PATH = SAVE_PATH / "tickers_universe.csv"
uni.to_csv(UNI_PATH, index=False)


Universe size: 100 (US=50, UK=50)


## Data Cleaning

Cleans and resamples data to month-end. Removes nulls and duplicates, then applies a liquidity screen to keep the most liquid stocks.


In [22]:
def download_benchmark(symbol: str):
    df = yf.download(symbol, start=START, end=END, interval=FREQ, auto_adjust=AUTO_ADJUST)
    df.to_csv(SAVE_PATH / f"{symbol.strip('^').lower()}_{FREQ}_data.csv")
    print(f"{symbol} downloaded: {len(df)} rows")
    return df

def download_in_batches(tickers, batch=25):
    frames = []
    for i in range(0, len(tickers), batch):
        chunk = tickers[i:i+batch]
        raw = yf.download(chunk, start=START, end=END, interval=FREQ,
                          auto_adjust=AUTO_ADJUST, group_by="ticker", progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            present = set(raw.columns.get_level_values(0))
            missing = [t for t in chunk if t not in present]
            for t in missing:
                print(f"[warn] no data for {t}")
            for t in (set(chunk) - set(missing)):
                df_t = raw[t].copy()
                df_t["ticker"] = t
                frames.append(df_t.reset_index().rename(columns=str.lower))
        else:
            # Single-ticker fallback
            if raw.empty:
                print(f"[warn] empty data for {chunk[0]}")
            else:
                raw["ticker"] = chunk[0]
                frames.append(raw.reset_index().rename(columns=str.lower))
    if not frames:
        return pd.DataFrame(columns=["date","open","high","low","close","adj close","volume","ticker"])
    return pd.concat(frames, ignore_index=True)

# Benchmarks
snp_data  = download_benchmark(SNP)
ftse_data = download_benchmark(FTSE)

# Basket
tickers = uni["ticker"].tolist()
basket_raw = download_in_batches(tickers, batch=25)
basket_raw.to_csv(SAVE_PATH / "equities_basket_1mo_data.csv", index=False)
print(f"Equities basket downloaded: {len(basket_raw)} rows")


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


^GSPC downloaded: 249 rows
^FTSE downloaded: 249 rows
Equities basket downloaded: 24900 rows


## Data Download

I download benchmark indices first (S&P 500 and FTSE), then download stock data in batches to avoid API rate limits. The download function handles missing tickers gracefully and saves everything to CSV files.


In [24]:
datasets = {
    "SNP500": snp_data,
    "FTSE": ftse_data,
    "Tidy Equities Basket": basket_raw
}

print("=== Clean & merge meta ===")
tb = datasets["Tidy Equities Basket"].copy()
tb["date"] = pd.to_datetime(tb["date"], errors="coerce")
tb = tb.dropna(subset=["date"])
# Keep essential columns
for c in ["close","volume","ticker"]:
    if c not in tb.columns:
        tb[c] = pd.NA
tb = tb[["date","close","volume","ticker"]]
# merge country/sector
tb = tb.merge(uni, on="ticker", how="left")

# Drop null close rows
nulls = tb["close"].isna().sum()
if nulls:
    before = len(tb); tb = tb.dropna(subset=["close"]); after = len(tb)
    print(f"Tidy basket: dropped {before-after} rows with null close")

# Deduplicate & sort
dups = tb.duplicated(subset=["ticker","date"]).sum()
if dups:
    before = len(tb); tb = tb.drop_duplicates(subset=["ticker","date"], keep="last"); after = len(tb)
    print(f"Tidy basket: dropped {before-after} duplicate (ticker,date) rows")
tb = tb.sort_values(["ticker","date"])

# Month-end resample per ticker
resampled = []
for tkr, g in tb.groupby("ticker", sort=False):
    g = g.set_index("date").sort_index()
    m = g.resample("ME").last()
    m["ticker"] = tkr
    meta = uni.loc[uni["ticker"]==tkr, ["country","sector"]].iloc[0]
    m["country"] = meta["country"]; m["sector"] = meta["sector"]
    resampled.append(m.reset_index())
tb_m = pd.concat(resampled, ignore_index=True)
datasets["Tidy Equities Basket"] = tb_m

# Benchmarks: month-end last, sorted, no duplicate index
for name in ["SNP500","FTSE"]:
    d = datasets[name]
    d = d[~d.index.duplicated(keep="last")].sort_index()
    datasets[name] = d.resample("ME").last()

print("\n Save to interim (same filenames as before) ")
datasets['SNP500'].to_csv(INTERIM_PATH / "snp500_1mo_cleaned_data.csv")
datasets['FTSE'].to_csv(INTERIM_PATH / "ftse_1mo_cleaned_data.csv")
datasets['Tidy Equities Basket'].to_csv(INTERIM_PATH / "equities_1mo_cleaned_data.csv", index=False)
print("Data saved successfully")

print("\n Final Summary ")
for name, df in datasets.items():
    n = len(df)
    nn = df.isna().sum().sum() if name=="Tidy Equities Basket" else df.isna().sum().sum()
    print(f"{name}: {n} rows, nulls={nn}")


=== Clean & merge meta ===
Tidy basket: dropped 671 rows with null close

 Save to interim (same filenames as before) 
Data saved successfully

 Final Summary 
SNP500: 249 rows, nulls=0
FTSE: 249 rows, nulls=0
Tidy Equities Basket: 24229 rows, nulls=0


In [27]:
# Drop illiquid tail per ticker using median monthly volume
medvol = tb_m.groupby("ticker")["volume"].median().sort_values(ascending=False)
keep = medvol.index[:min(100, len(medvol))]  # cap at ~100
tb_m = tb_m[tb_m["ticker"].isin(keep)].copy()
datasets["Tidy Equities Basket"] = tb_m
print(f"Liquidity screen → kept {len(keep)} tickers")


Liquidity screen → kept 100 tickers


In [29]:
datasets["Tidy Equities Basket"].to_parquet(INTERIM_PATH / "equities_1mo_cleaned_data_1b.parquet", index=False)
